In [1]:
!pip install "openai<1.0.0"


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Install required packages
import subprocess
import sys

packages = ['numpy', 'scikit-learn', 'openai', 'python-dotenv']
for package in packages:
    try:
        __import__(package)
    except ImportError:
        print(f'Installing {package}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

print('All dependencies installed successfully!')

Installing scikit-learn...
Installing python-dotenv...
All dependencies installed successfully!


In [3]:
# Core imports
import os
import json
import gzip
import time
import random
import pickle
import logging
import tempfile
import subprocess
import ast
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, field
from functools import wraps
from pathlib import Path
from collections import deque

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv
import openai

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] %(message)s'
)
logger = logging.getLogger(__name__)

print('✓ All imports successful')

✓ All imports successful


In [4]:
class SecureConfigLoader:
    """Load and validate configuration from .env file."""
    
    def __init__(self):
        self.config = {}
    
    def load_from_env_file(self, env_path: str = '.env') -> Dict[str, str]:
        """Load configuration from .env file."""
        if not os.path.exists(env_path):
            logger.warning(f'{env_path} not found. Creating template...')
            self._create_template(env_path)
            raise FileNotFoundError(
                f'Please fill {env_path} with your API credentials and run again.'
            )
        
        load_dotenv(env_path)
        self.config = {
            'openrouter_api_key': os.getenv('OPENROUTER_API_KEY'),
            'openrouter_model': os.getenv('OPENROUTER_MODEL', 'google/gemini-2.0-flash-exp:free'),
            'gemini_api_base': os.getenv('GEMINI_API_BASE', 'https://openrouter.ai/api/v1/'),
        }
        
        self.validate_config()
        logger.info('✓ Configuration loaded successfully')
        return self.config
    
    def validate_config(self):
        """Validate that required config values are present."""
        required_keys = ['openrouter_api_key']
        missing = [key for key in required_keys if not self.config.get(key)]
        
        if missing:
            raise ValueError(f'Missing required config keys: {missing}')
    
    @staticmethod
    def _create_template(env_path: str):
        """Create .env template file."""
        template = '''OPENROUTER_API_KEY=sk-or-v1-YOUR-KEY-HERE
OPENROUTER_MODEL=google/gemini-2.0-flash-exp:free
GEMINI_API_BASE=https://openrouter.ai/api/v1/
'''
        with open(env_path, 'w') as f:
            f.write(template)

# Test configuration loading
print('✓ SecureConfigLoader class defined')

✓ SecureConfigLoader class defined


In [5]:
def exponential_backoff(
    max_retries: int = 5,
    initial_delay: float = 1.0,
    backoff_factor: float = 2.0,
    jitter: bool = True
):
    """Decorator for exponential backoff with jitter for API rate limiting.
    
    Args:
        max_retries: Maximum number of retry attempts
        initial_delay: Initial delay in seconds
        backoff_factor: Multiplier for delay after each retry
        jitter: Whether to add random jitter to delay
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            delay = initial_delay
            last_exception = None
            
            for attempt in range(max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    
                    # Check if it's a rate limit error (429)
                    if attempt < max_retries and ('429' in str(e) or 'rate' in str(e).lower()):
                        # Add jitter to prevent thundering herd
                        jitter_amount = random.uniform(0, delay * 0.1) if jitter else 0
                        wait_time = delay + jitter_amount
                        
                        logger.info(
                            f'Rate limited. Retrying in {wait_time:.2f}s '
                            f'(attempt {attempt + 1}/{max_retries})'
                        )
                        time.sleep(wait_time)
                        delay *= backoff_factor
                    else:
                        raise
            
            raise last_exception
        return wrapper
    return decorator

print('✓ exponential_backoff decorator defined')

✓ exponential_backoff decorator defined


In [19]:
import openai
import requests
from typing import Optional
import numpy as np

class BaseLLMModel:
    """OpenRouter-compatible LLM model."""

    def __init__(
        self,
        api_key: str,
        model: str = 'google/gemini-2.0-flash-exp:free',
        api_base: str = 'https://openrouter.ai/api/v1/',
        temperature: float = 0.1  # From your .env
    ):
        self.api_key = api_key
        self.model = model
        self.api_base = api_base
        self.temperature = temperature
        
        # OpenRouter headers
        self.headers = {
            'Authorization': f'Bearer {api_key}',
            'HTTP-Referer': 'https://your-app.com',  # Required by OpenRouter
            'X-Title': 'Reflexion Ablation Study'     # Required by OpenRouter
        }

    @exponential_backoff(max_retries=5, initial_delay=1.0)
    def call_llm(
        self,
        prompt: str,
        max_tokens: int = 2048,
        temperature: Optional[float] = None
    ) -> str:
        """Call OpenRouter chat completions."""
        temp = temperature if temperature is not None else self.temperature
        
        payload = {
            'model': self.model,
            'messages': [{'role': 'user', 'content': prompt}],
            'max_tokens': max_tokens,
            'temperature': temp
        }
        
        response = requests.post(
            f"{self.api_base}chat/completions",
            headers=self.headers,
            json=payload
        )
        response.raise_for_status()
        
        data = response.json()
        return data['choices'][0]['message']['content']

    @exponential_backoff(max_retries=5, initial_delay=1.0)
    def get_embedding(self, text: str) -> np.ndarray:
        """Get embeddings (OpenRouter supports OpenAI format)."""
        payload = {
            'model': 'text-embedding-ada-002',
            'input': text
        }
        
        response = requests.post(
            f"{self.api_base}embeddings",
            headers=self.headers,
            json=payload
        )
        response.raise_for_status()
        
        data = response.json()
        return np.array(data['data'][0]['embedding'])

print('✓ BaseLLMModel FIXED (OpenRouter compatible)')


✓ BaseLLMModel FIXED (OpenRouter compatible)


In [7]:
class TemporalMemory:
    """Mode A: Baseline temporal/sliding window memory."""
    
    def __init__(self, max_size: int = 10):
        """Initialize temporal memory.
        
        Args:
            max_size: Maximum number of reflections to store
        """
        self.max_size = max_size
        self.reflections = deque(maxlen=max_size)
    
    def add_reflection(self, reflection: str) -> None:
        """Add reflection to memory.
        
        Args:
            reflection: Reflection text
        """
        self.reflections.append(reflection)
        logger.debug(f'Added reflection: {reflection[:50]}...')
    
    def get_relevant_memories(self, k: int = 2) -> List[str]:
        """Get last k reflections (temporal/recency-based).
        
        Args:
            k: Number of reflections to retrieve
        
        Returns:
            List of k most recent reflections
        """
        return list(self.reflections)[-k:]
    
    def save(self, path: str) -> None:
        """Save memory to disk."""
        with open(path, 'wb') as f:
            pickle.dump(list(self.reflections), f)
    
    def load(self, path: str) -> None:
        """Load memory from disk."""
        with open(path, 'rb') as f:
            data = pickle.load(f)
            self.reflections.clear()
            for item in data:
                self.reflections.append(item)
    
    def clear(self) -> None:
        """Clear all memories."""
        self.reflections.clear()


class VectorEpisodicMemory:
    """Mode B: Vector episodic memory with semantic search."""
    
    def __init__(
        self,
        api_key: str,
        api_base: str = 'https://api.openai.com/v1/',
        max_size: int = 100,
        similarity_threshold: float = 0.5
    ):
        """Initialize vector memory.
        
        Args:
            api_key: OpenAI API key for embeddings
            api_base: API base URL
            max_size: Maximum number of reflections
            similarity_threshold: Minimum similarity for relevance
        """
        self.llm = BaseLLMModel(api_key=api_key, api_base=api_base)
        self.max_size = max_size
        self.similarity_threshold = similarity_threshold
        self.reflections = deque(maxlen=max_size)
        self.embeddings = deque(maxlen=max_size)
    
    def add_reflection(self, reflection: str) -> None:
        """Add reflection with embedding to memory.
        
        Args:
            reflection: Reflection text
        """
        embedding = self.llm.get_embedding(reflection)
        self.reflections.append(reflection)
        self.embeddings.append(embedding)
        logger.debug(f'Added reflection with embedding: {reflection[:50]}...')
    
    def get_relevant_memories(self, query: str, k: int = 2) -> List[str]:
        """Get top-k semantically similar reflections.
        
        Args:
            query: Query text to find similar reflections
            k: Number of reflections to retrieve
        
        Returns:
            List of k most similar reflections
        """
        if not self.reflections:
            return []
        
        # Get query embedding
        query_embedding = self.llm.get_embedding(query)
        
        # Compute similarities
        embeddings_array = np.array(list(self.embeddings))
        similarities = cosine_similarity(
            query_embedding.reshape(1, -1),
            embeddings_array
        )[0]
        
        # Get top-k above threshold
        valid_indices = np.where(similarities >= self.similarity_threshold)[0]
        if len(valid_indices) == 0:
            valid_indices = np.argsort(similarities)[-k:]
        else:
            valid_indices = valid_indices[np.argsort(similarities[valid_indices])[-k:]]
        
        relevant = [list(self.reflections)[i] for i in sorted(valid_indices)]
        logger.debug(f'Retrieved {len(relevant)} similar reflections')
        return relevant
    
    def save(self, path: str) -> None:
        """Save memory to disk."""
        data = {
            'reflections': list(self.reflections),
            'embeddings': [e.tolist() for e in self.embeddings]
        }
        with open(path, 'wb') as f:
            pickle.dump(data, f)
    
    def load(self, path: str) -> None:
        """Load memory from disk."""
        with open(path, 'rb') as f:
            data = pickle.load(f)
            self.reflections.clear()
            self.embeddings.clear()
            for refl, emb in zip(data['reflections'], data['embeddings']):
                self.reflections.append(refl)
                self.embeddings.append(np.array(emb))
    
    def clear(self) -> None:
        """Clear all memories."""
        self.reflections.clear()
        self.embeddings.clear()

print('✓ TemporalMemory and VectorEpisodicMemory classes defined')

✓ TemporalMemory and VectorEpisodicMemory classes defined


In [8]:
class ObjectiveCodeEvaluator:
    """Evaluate generated code with subprocess sandboxing."""
    
    def __init__(self, timeout: int = 5):
        """Initialize code evaluator.
        
        Args:
            timeout: Execution timeout in seconds
        """
        self.timeout = timeout
    
    def evaluate(
        self,
        code: str,
        test_cases: List[Tuple[Dict[str, Any], Any]]
    ) -> Dict[str, Any]:
        """Evaluate code against test cases.
        
        Args:
            code: Generated code to evaluate
            test_cases: List of (input_dict, expected_output) tuples
        
        Returns:
            Evaluation results dict
        """
        # Validate syntax
        try:
            ast.parse(code)
        except SyntaxError as e:
            return {
                'passed': 0,
                'total': len(test_cases),
                'pass_rate': 0.0,
                'status': 'syntax_error',
                'error': str(e)
            }
        
        # Create test driver
        passed = 0
        errors = []
        
        for test_input, expected_output in test_cases:
            try:
                # Create temporary test file
                with tempfile.NamedTemporaryFile(
                    mode='w',
                    suffix='.py',
                    delete=False
                ) as f:
                    # Write code + test
                    f.write(code)
                    f.write('\n\n')
                    f.write('import json\n')
                    f.write('try:\n')
                    f.write(f'    result = solve(**{test_input})\n')
                    f.write(f'    expected = {expected_output}\n')
                    f.write('    assert result == expected, f"Got {result}, expected {expected}"\n')
                    f.write('    print("PASS")\n')
                    f.write('except Exception as e:\n')
                    f.write('    print(f"FAIL: {str(e)}")\n')
                    temp_path = f.name
                
                # Execute with timeout
                result = subprocess.run(
                    [sys.executable, temp_path],
                    capture_output=True,
                    text=True,
                    timeout=self.timeout
                )
                
                if 'PASS' in result.stdout:
                    passed += 1
                else:
                    errors.append(result.stdout or result.stderr)
                
                # Cleanup
                os.unlink(temp_path)
            
            except subprocess.TimeoutExpired:
                errors.append(f'Timeout after {self.timeout}s')
            except Exception as e:
                errors.append(str(e))
        
        return {
            'passed': passed,
            'total': len(test_cases),
            'pass_rate': passed / len(test_cases) if test_cases else 0.0,
            'status': 'success',
            'errors': errors
        }


class HumanEvalLoader:
    """Load and manage HumanEval benchmark tasks."""
    
    @staticmethod
    def load_from_file(
        file_path: str,
        num_samples: Optional[int] = None
    ) -> List[Dict[str, Any]]:
        """Load HumanEval tasks from .jsonl.gz file.
        
        Args:
            file_path: Path to HumanEval.jsonl.gz
            num_samples: Limit number of tasks (None = all)
        
        Returns:
            List of task dicts
        """
        tasks = []
        try:
            with gzip.open(file_path, 'rt') as f:
                for i, line in enumerate(f):
                    if num_samples and i >= num_samples:
                        break
                    tasks.append(json.loads(line))
        except FileNotFoundError:
            logger.warning(f'{file_path} not found. Using synthetic tasks.')
            return HumanEvalLoader.create_synthetic_tasks(num_samples or 5)
        
        logger.info(f'Loaded {len(tasks)} tasks from {file_path}')
        return tasks
    
    @staticmethod
    def create_synthetic_tasks(num_tasks: int = 5) -> List[Dict[str, Any]]:
        """Create synthetic programming tasks for testing.
        
        Args:
            num_tasks: Number of synthetic tasks to create
        
        Returns:
            List of synthetic task dicts
        """
        tasks = [
            {
                'task_id': 'HumanEval/0',
                'prompt': 'def reverse_string(s: str) -> str:\n    """Reverse a string."""',
                'test_cases': [('hello', 'olleh'), ('world', 'dlrow')]
            },
            {
                'task_id': 'HumanEval/1',
                'prompt': 'def count_vowels(s: str) -> int:\n    """Count vowels in string."""',
                'test_cases': [('hello', 2), ('aeiou', 5)]
            },
            {
                'task_id': 'HumanEval/2',
                'prompt': 'def factorial(n: int) -> int:\n    """Compute factorial of n."""',
                'test_cases': [('0', 1), ('5', 120)]
            },
            {
                'task_id': 'HumanEval/3',
                'prompt': 'def is_palindrome(s: str) -> bool:\n    """Check if string is palindrome."""',
                'test_cases': [('racecar', True), ('hello', False)]
            },
            {
                'task_id': 'HumanEval/4',
                'prompt': 'def find_max(arr: list) -> int:\n    """Find maximum element in list."""',
                'test_cases': [('[1, 5, 3, 2]', 5), ('[10, 2, 8]', 10)]
            },
        ]
        return tasks[:num_tasks]

print('✓ ObjectiveCodeEvaluator and HumanEvalLoader classes defined')

✓ ObjectiveCodeEvaluator and HumanEvalLoader classes defined


In [9]:
class ReflexionAgent:
    """Reflexion agent with pluggable memory backends."""
    
    def __init__(
        self,
        llm: BaseLLMModel,
        memory: Optional[object] = None,
        memory_mode: str = 'vector',
        api_key: Optional[str] = None,
        api_base: str = 'https://api.openai.com/v1/',
        max_trials: int = 5
    ):
        """Initialize Reflexion agent.
        
        Args:
            llm: BaseLLMModel instance
            memory: Optional pre-initialized memory object
            memory_mode: 'temporal' or 'vector'
            api_key: API key for vector memory
            api_base: API base URL
            max_trials: Maximum attempts per task
        """
        self.llm = llm
        self.max_trials = max_trials
        self.evaluator = ObjectiveCodeEvaluator(timeout=5)
        
        # Initialize memory
        if memory is None:
            if memory_mode == 'temporal':
                self.memory = TemporalMemory(max_size=10)
            elif memory_mode == 'vector':
                self.memory = VectorEpisodicMemory(
                    api_key=api_key or llm.api_key,
                    api_base=api_base,
                    max_size=100
                )
            else:
                raise ValueError(f'Unknown memory_mode: {memory_mode}')
        else:
            self.memory = memory
        
        self.memory_mode = memory_mode
    
    def solve_task(
        self,
        task: Dict[str, Any],
        verbose: bool = False
    ) -> Dict[str, Any]:
        """Solve a programming task with reflection.
        
        Args:
            task: Task dict with 'prompt' and 'test_cases'
            verbose: Print detailed logs
        
        Returns:
            Results dict
        """
        task_id = task.get('task_id', 'unknown')
        
        for trial in range(self.max_trials):
            # Get relevant memories
            if self.memory_mode == 'temporal':
                memories = self.memory.get_relevant_memories(k=2)
            else:  # vector
                memories = self.memory.get_relevant_memories(
                    query=task['prompt'],
                    k=2
                )
            
            # Construct prompt with memories
            memory_context = '\n'.join(f'- {m}' for m in memories) if memories else 'None'
            prompt = f"""Task: {task['prompt']}

Previous reflections:
{memory_context}

Generate Python code to solve this task. Include function definition and implementation.
"""
            
            # Generate code
            try:
                code = self.llm.call_llm(prompt, max_tokens=1024, temperature=0.7)
                
                # Extract function if wrapped in markdown
                if '```python' in code:
                    code = code.split('```python')[1].split('```')[0].strip()
                elif '```' in code:
                    code = code.split('```')[1].split('```')[0].strip()
                
                # Evaluate code
                results = self.evaluator.evaluate(code, task['test_cases'])
                
                if results['pass_rate'] == 1.0:
                    if verbose:
                        logger.info(f'✓ Task {task_id} solved on trial {trial + 1}')
                    return {
                        'task_id': task_id,
                        'success': True,
                        'trials': trial + 1,
                        'code': code
                    }
                else:
                    # Reflect on failure
                    reflection = f"Failed: {results['errors'][0] if results['errors'] else 'Wrong output'}"
                    self.memory.add_reflection(reflection)
                    if verbose:
                        logger.info(f'Trial {trial + 1} failed: {reflection}')
            
            except Exception as e:
                reflection = f"Error: {str(e)}"
                self.memory.add_reflection(reflection)
                if verbose:
                    logger.info(f'Trial {trial + 1} error: {reflection}')
        
        # Max trials reached
        if verbose:
            logger.info(f'✗ Task {task_id} failed after {self.max_trials} trials')
        return {
            'task_id': task_id,
            'success': False,
            'trials': self.max_trials
        }
    
    def reset(self):
        """Reset agent memory."""
        self.memory.clear()

print('✓ ReflexionAgent class defined')

✓ ReflexionAgent class defined


In [20]:
class AblationStudy:
    """Run comparative ablation studies between memory modes."""
    
    def __init__(self, config: Dict[str, str]):
        """Initialize ablation study.
        
        Args:
            config: Configuration dict with API credentials
        """
        self.config = config
        self.llm = BaseLLMModel(
            api_key=config['openrouter_api_key'],
            model=config['openrouter_model'],
            api_base=config['gemini_api_base']
        )
        self.results = []
    
    def run_benchmark(
        self,
        tasks: List[Dict[str, Any]],
        modes: List[str] = None
    ) -> List[Dict[str, Any]]:
        """Run ablation study on tasks.
        
        Args:
            tasks: List of tasks to evaluate
            modes: List of memory modes to test
        
        Returns:
            Results list
        """
        if modes is None:
            modes = ['temporal', 'vector']
        
        self.results = []
        
        for mode in modes:
            logger.info(f'\n{'='*60}')
            logger.info(f'Running Mode: {mode.upper()}')
            logger.info(f'{'='*60}')
            
            # Create agent for this mode
            agent = ReflexionAgent(
                llm=self.llm,
                memory_mode=mode,
                api_key=self.config['openrouter_api_key'],
                api_base=self.config['gemini_api_base']
            )
            
            # Run on each task
            for task in tasks:
                result = agent.solve_task(task, verbose=True)
                result['mode'] = mode
                self.results.append(result)
                
                logger.info(f"Task: {result['task_id']} | "
                           f"Success: {result['success']} | "
                           f"Trials: {result['trials']}")
            
            # Reset for next mode
            agent.reset()
        
        return self.results
    
    def generate_summary_table(self) -> str:
        """Generate summary table of results.
        
        Returns:
            Formatted table string
        """
        if not self.results:
            return "No results to display"
        
        # Group by mode
        modes = {}
        for result in self.results:
            mode = result['mode']
            if mode not in modes:
                modes[mode] = []
            modes[mode].append(result)
        
        # Calculate statistics
        table = '\n' + '='*80 + '\n'
        table += 'ABLATION STUDY RESULTS: Temporal (Mode A) vs Vector (Mode B) Memory\n'
        table += '='*80 + '\n'
        table += f'{"Task ID":<20} | {"Mode":<10} | {"Pass@1":<10} | {"Avg Trials":<12}\n'
        table += '-'*80 + '\n'
        
        task_ids = set(r['task_id'] for r in self.results)
        
        for task_id in sorted(task_ids):
            task_results = [r for r in self.results if r['task_id'] == task_id]
            
            for mode in sorted(modes.keys()):
                mode_results = [r for r in task_results if r['mode'] == mode]
                if mode_results:
                    result = mode_results[0]
                    pass_at_1 = '✓' if result['success'] else '✗'
                    table += (f"{task_id:<20} | {mode:<10} | "
                             f"{pass_at_1:<10} | {result['trials']:<12}\n")
        
        table += '-'*80 + '\n\n'
        
        # Summary statistics
        table += '📊 SUMMARY STATISTICS:\n'
        for mode in sorted(modes.keys()):
            mode_results = modes[mode]
            pass_count = sum(1 for r in mode_results if r['success'])
            pass_rate = (pass_count / len(mode_results) * 100) if mode_results else 0
            avg_trials = (sum(r['trials'] for r in mode_results) / 
                          len(mode_results) if mode_results else 0)
            
            table += f"  {mode.upper():<10}: Pass@1={pass_rate:.1f}% | Avg Trials={avg_trials:.2f}\n"
        
        # Improvement
        if 'temporal' in modes and 'vector' in modes:
            temporal_results = modes['temporal']
            vector_results = modes['vector']
            
            temporal_pass = sum(1 for r in temporal_results if r['success']) / len(temporal_results)
            vector_pass = sum(1 for r in vector_results if r['success']) / len(vector_results)
            improvement = ((vector_pass - temporal_pass) / temporal_pass * 100) if temporal_pass > 0 else 0
            
            temporal_trials = sum(r['trials'] for r in temporal_results) / len(temporal_results)
            vector_trials = sum(r['trials'] for r in vector_results) / len(vector_results)
            trials_improvement = ((temporal_trials - vector_trials) / temporal_trials * 100) if temporal_trials > 0 else 0
            
            table += f"\n  IMPROVEMENT: Pass@1={improvement:+.1f}% | Trials={trials_improvement:+.1f}%\n"
        
        table += '='*80 + '\n'
        
        return table

print('✓ AblationStudy class defined')

✓ AblationStudy class defined


In [21]:
# Load configuration
config_loader = SecureConfigLoader()
config = config_loader.load_from_env_file('.env')

print('\n✓ Configuration loaded')
print(f'  Model: {config["openrouter_model"]}')
print(f'  API Base: {config["gemini_api_base"]}')

[INFO] ✓ Configuration loaded successfully



✓ Configuration loaded
  Model: google/gemini-2.0-flash-exp:free
  API Base: https://openrouter.ai/api/v1/


In [23]:
# Create synthetic tasks for testing
tasks = HumanEvalLoader.create_synthetic_tasks(num_tasks=5)

print(f'\n✓ Created {len(tasks)} synthetic tasks')
for task in tasks:
    print(f'  - {task["task_id"]}: {task["prompt"][:50]}...')


✓ Created 5 synthetic tasks
  - HumanEval/0: def reverse_string(s: str) -> str:
    """Reverse ...
  - HumanEval/1: def count_vowels(s: str) -> int:
    """Count vowe...
  - HumanEval/2: def factorial(n: int) -> int:
    """Compute facto...
  - HumanEval/3: def is_palindrome(s: str) -> bool:
    """Check if...
  - HumanEval/4: def find_max(arr: list) -> int:
    """Find maximu...


In [ ]:
# Run ablation study
print('\nStarting ablation study...')
print('This will compare Temporal (Mode A) vs Vector (Mode B) memory')
print('Expected time: 10-15 minutes for 5 tasks\n')

ablation = AblationStudy(config)
results = ablation.run_benchmark(tasks, modes=['temporal', 'vector'])

print('\n✓ Ablation study complete')

[INFO] 
[INFO] Running Mode: TEMPORAL
[INFO] ============================================================



Starting ablation study...
This will compare Temporal (Mode A) vs Vector (Mode B) memory
Expected time: 10-15 minutes for 5 tasks



[INFO] Rate limited. Retrying in 1.08s (attempt 1/5)
[INFO] Rate limited. Retrying in 2.13s (attempt 2/5)
[INFO] Rate limited. Retrying in 4.06s (attempt 3/5)
[INFO] Rate limited. Retrying in 8.26s (attempt 4/5)
[INFO] Rate limited. Retrying in 16.93s (attempt 5/5)
[INFO] Trial 1 error: Error: 429 Client Error: Too Many Requests for url: https://openrouter.ai/api/v1/chat/completions
[INFO] Rate limited. Retrying in 1.08s (attempt 1/5)
[INFO] Rate limited. Retrying in 2.07s (attempt 2/5)
[INFO] Rate limited. Retrying in 4.35s (attempt 3/5)
[INFO] Rate limited. Retrying in 8.60s (attempt 4/5)
[INFO] Trial 2 failed: Failed: FAIL: name 'solve' is not defined

[INFO] Rate limited. Retrying in 1.09s (attempt 1/5)
[INFO] Rate limited. Retrying in 2.15s (attempt 2/5)
[INFO] Rate limited. Retrying in 4.36s (attempt 3/5)
[INFO] Rate limited. Retrying in 8.12s (attempt 4/5)
[INFO] Rate limited. Retrying in 16.00s (attempt 5/5)
[INFO] Trial 3 error: Error: 429 Client Error: Too Many Requests for u

: 

In [ ]:
# Display results
summary_table = ablation.generate_summary_table()
print(summary_table)

In [ ]:
# --- ASSUMING SecureConfigLoader is defined in Cell 27 ---
import os # Make sure this import is at the top of the notebook

# Load config
config_loader = SecureConfigLoader()
config_loader.load_from_env_file(".env") 

# --- CRITICAL MANUAL OVERRIDE FOR OPENROUTER BASE URL ---
# This ensures the API base URL is set for the Google SDK, even if the 
# .env loading was incomplete for this specific variable.
os.environ["GEMINI_API_BASE"] = "https://openrouter.ai/api/v1/"
# ---------------------------------------------------------

if config_loader.api_key:
    print(f"✅ Configuration loaded successfully. API key found: {config_loader.api_key[:4]}...{config_loader.api_key[-4:]}")
else:
    print("🛑 ERROR: API key is still missing. Please ensure your .env file is correct.")

# Now, config_loader is defined and ready to be used by the agent.

In [ ]:
agent = ReflexionAgent(
    config_loader=config_loader,
    max_trials=5,        # ✅ CHANGED: from 3 to 5
    memory_size=3,       # ✅ CHANGED: from 2 to 3
    temperature=0.7
)

print("✅ Reflexion agent initialized (5 trials max)")

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 1: DECISION-MAKING WITH SELF-REFLECTION")
print("="*80)

decision_with_reflection = []

for task_def in ExperimentSuite.decision_making_tasks():
    print(f"\n📋 Task: {task_def['id']}")
    
    # --- CRITICAL FIX: Change method name and argument passing ---
    # We now call solve_decision_task and pass the full task_def dictionary.
    result = agent.solve_decision_task(
        task_def,  
        success_threshold=85,
        verbose=True,
        show_reflection=True
    )
    
    decision_with_reflection.append({
        "task_id": task_def['id'],
        "mode": "WITH_REFLECTION",
        "success": result['success'],
        "trials": result['trials'],
        "score": result['final_score'],
        "api_calls": result['api_calls'],
        "time": result['total_time'],
        "memory_size": result['final_memory_size']
    })
    
    agent.reset()

    print("⏳ Cooling down for 10 seconds to avoid Rate Limit...")
    time.sleep(10)
    
print(f"\n✓ Decision-Making Results (WITH reflection): {len([r for r in decision_with_reflection if r['success']])}/{len(decision_with_reflection)} successful")


In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 1B: DECISION-MAKING WITHOUT REFLECTION (BASELINE)")
print("="*80)

decision_without_reflection = []

for task_def in ExperimentSuite.decision_making_tasks():
    print(f"\n📋 Task: {task_def['id']} (NO REFLECTION)")
    result = agent.solve_task(
        task_def['task'],
        success_threshold=85,  # ✅ CHANGED: from 75 to 85
        verbose=True,
        show_reflection=False
    )
    
    decision_without_reflection.append({
        "task_id": task_def['id'],
        "mode": "WITHOUT_REFLECTION",
        "success": result['success'],
        "trials": result['trials'],
        "score": result['final_score'],
        "api_calls": result['api_calls'],
        "time": result['total_time'],
        "memory_size": result['final_memory_size']
    })
    
    agent.reset()

print(f"\n✓ Decision-Making Results (WITHOUT reflection): {len([r for r in decision_without_reflection if r['success']])}/{len(decision_without_reflection)} successful")


In [ ]:
print("\n" + "="*80)
print("COMPARISON: WITH vs WITHOUT SELF-REFLECTION (Decision-Making)")
print("="*80)

print("\n┌─────────────────────┬──────────────┬────────┬──────────┬──────────────┐")
print("│ Mode                │ Task         │ Trials │ Score    │ Success      │")
print("├─────────────────────┼──────────────┼────────┼──────────┼──────────────┤")

for r1 in decision_with_reflection:
    print(f"│ WITH Reflection     │ {r1['task_id']:<12} │ {r1['trials']:<6} │ {r1['score']:<8} │ {'✓' if r1['success'] else '✗':<12} │")

for r2 in decision_without_reflection:
    print(f"│ WITHOUT Reflection  │ {r2['task_id']:<12} │ {r2['trials']:<6} │ {r2['score']:<8} │ {'✓' if r2['success'] else '✗':<12} │")

print("└─────────────────────┴──────────────┴────────┴──────────┴──────────────┘")


In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 2: REASONING WITH SELF-REFLECTION")
print("="*80)

reasoning_with_reflection = []

for task_def in ExperimentSuite.reasoning_tasks():
    print(f"\n🧠 Task: {task_def['id']}")
    reasoning_prompt = f"""You will answer a reasoning question based on context.

CONTEXT:
{task_def['context']}

QUESTION:
{task_def['question']}

Answer clearly and briefly, then explain your reasoning."""
    
    result = agent.solve_task(
        reasoning_prompt,
        success_threshold=85,  # keep 85
        verbose=True,
        show_reflection=True
    )

    
    reasoning_with_reflection.append({
        "task_id": task_def['id'],
        "mode": "WITH_REFLECTION",
        "success": result['success'],
        "trials": result['trials'],
        "score": result['final_score'],
        "api_calls": result['api_calls'],
        "time": result['total_time']
    })
    
    agent.reset()


In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 2B: REASONING WITHOUT REFLECTION (BASELINE)")
print("="*80)

reasoning_without_reflection = []

for task_def in ExperimentSuite.reasoning_tasks():
    print(f"\n🧠 Task: {task_def['id']} (NO REFLECTION)")

    reasoning_prompt = f"""You will answer a reasoning question based on context.

CONTEXT:
{task_def['context']}

QUESTION:
{task_def['question']}

Answer clearly and briefly, then explain your reasoning."""

    result = agent.solve_task(
        reasoning_prompt,
        success_threshold=85,  # keep 85
        verbose=True,
        show_reflection=False
    )
    
    reasoning_without_reflection.append({
        "task_id": task_def['id'],
        "mode": "WITHOUT_REFLECTION",
        "success": result['success'],
        "trials": result['trials'],
        "score": result['final_score'],
        "api_calls": result['api_calls'],
        "time": result['total_time']
    })
    
    agent.reset()


In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 3B: PROGRAMMING WITHOUT REFLECTION (BASELINE)")
print("="*80)

programming_without_reflection = []

for task_def in ExperimentSuite.programming_tasks():
    print(f"\n💻 Task: {task_def['id']} (NO REFLECTION)")

    programming_prompt = f"""You must solve a programming task.

TASK:
{task_def['prompt']}

REQUIREMENTS:
- Write clear, correct Python code.
- Include any necessary helper functions.
- Use good style and comments where useful.
"""
    # --- START OF MODIFICATION ---
    result = agent.solve_programming_task( # <-- Use the specialized method
        programming_prompt,
        test_cases=task_def['tests'],    # <-- Pass the list of test cases
        success_threshold=85,
        verbose=True,
        show_reflection=False
    )
    # --- END OF MODIFICATION ---
    
    programming_without_reflection.append({
        "task_id": task_def['id'],
        "mode": "WITHOUT_REFLECTION",
        "success": result['success'],
        "trials": result['trials'],
        "score": result['final_score'],
        "api_calls": result['api_calls'],
        "time": result['total_time']
    })
    
    agent.reset()

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 3: PROGRAMMING WITH SELF-REFLECTION")
print("="*80)

programming_with_reflection = []

for task_def in ExperimentSuite.programming_tasks():
    print(f"\n💻 Task: {task_def['id']}")

    programming_prompt = f"""You must solve a programming task.

TASK:
{task_def['prompt']}

REQUIREMENTS:
- Write clear, correct Python code.
- Include any necessary helper functions.
- Use good style and comments where useful.
"""

    # --- START OF MODIFICATION ---
    # 1. Change to the specialized method
    # 2. Add the test_cases argument
    result = agent.solve_programming_task(
        programming_prompt,
        test_cases=task_def['tests'], # Passes the predefined tests for objective evaluation
        success_threshold=85,
        verbose=True,
        show_reflection=True
    )
    # --- END OF MODIFICATION ---

    programming_with_reflection.append({
        "task_id": task_def['id'],
        "mode": "WITH_REFLECTION",
        "success": result['success'],
        "trials": result['trials'],
        "score": result['final_score'],
        "api_calls": result['api_calls'],
        "time": result['total_time']
    })

    agent.reset()

In [ ]:
print("\n" + "="*80)
print("REFLEXION FRAMEWORK - COMPREHENSIVE ANALYSIS (WITH STRICT EVALUATOR)")
print("="*80)

all_results_with = decision_with_reflection + reasoning_with_reflection + programming_with_reflection
all_results_without = decision_without_reflection + reasoning_without_reflection + programming_without_reflection

with_success = len([r for r in all_results_with if r['success']])
without_success = len([r for r in all_results_without if r['success']])
with_avg_trials = sum(r['trials'] for r in all_results_with) / len(all_results_with) if all_results_with else 0
without_avg_trials = sum(r['trials'] for r in all_results_without) / len(all_results_without) if all_results_without else 0
with_avg_score = sum(r['score'] for r in all_results_with) / len(all_results_with) if all_results_with else 0
without_avg_score = sum(r['score'] for r in all_results_without) / len(all_results_without) if all_results_without else 0

print("\n" + "="*80)
print("SUMMARY STATISTICS: WITH vs WITHOUT SELF-REFLECTION (STRICT EVALUATOR)")
print("="*80)

print(f"\n{'┌'+'─'*78+'┐'}")
print(f"│ {'METRIC':<25} │ {'WITH REFLECTION':<25} │ {'WITHOUT REFLECTION':<23} │")
print(f"{'├'+'─'*78+'┤'}")
print(f"│ {'Success Rate':<25} │ {f'{with_success}/{len(all_results_with)}':<25} │ {f'{without_success}/{len(all_results_without)}':<23} │")
print(f"│ {'Average Trials':<25} │ {f'{with_avg_trials:.2f}':<25} │ {f'{without_avg_trials:.2f}':<23} │")
print(f"│ {'Average Score':<25} │ {f'{with_avg_score:.1f}/100':<25} │ {f'{without_avg_score:.1f}/100':<23} │")
print(f"{'└'+'─'*78+'┘'}\n")

if without_avg_trials > 0:
    trials_improvement = ((without_avg_trials - with_avg_trials) / without_avg_trials) * 100
else:
    trials_improvement = 0

score_improvement = with_avg_score - without_avg_score

print(f"{'▶ REFLEXION IMPROVEMENTS:':<40}")
print(f"  • Trials reduction: {trials_improvement:.1f}%")
print(f"  • Score improvement: +{score_improvement:.1f} points")
print(f"  • Success rate: {with_success}/{len(all_results_with)} vs {without_success}/{len(all_results_without)}")



In [ ]:

print("\n" + "="*80)
print("REFLEXION FRAMEWORK - COMPREHENSIVE ANALYSIS")
print("="*80)

# Collect all results
all_results_with = decision_with_reflection + reasoning_with_reflection + programming_with_reflection
all_results_without = decision_without_reflection + reasoning_without_reflection + programming_without_reflection

# Statistics
with_success = len([r for r in all_results_with if r['success']])
without_success = len([r for r in all_results_without if r['success']])
with_avg_trials = sum(r['trials'] for r in all_results_with) / len(all_results_with) if all_results_with else 0
without_avg_trials = sum(r['trials'] for r in all_results_without) / len(all_results_without) if all_results_without else 0
with_avg_score = sum(r['score'] for r in all_results_with) / len(all_results_with) if all_results_with else 0
without_avg_score = sum(r['score'] for r in all_results_without) / len(all_results_without) if all_results_without else 0

print("\n" + "="*80)
print("SUMMARY STATISTICS: WITH vs WITHOUT SELF-REFLECTION")
print("="*80)

print(f"\n{'┌'+'─'*78+'┐'}")
print(f"│ {'METRIC':<25} │ {'WITH REFLECTION':<25} │ {'WITHOUT REFLECTION':<23} │")
print(f"{'├'+'─'*78+'┤'}")
print(f"│ {'Success Rate':<25} │ {f'{with_success}/{len(all_results_with)}':<25} │ {f'{without_success}/{len(all_results_without)}':<23} │")
print(f"│ {'Average Trials':<25} │ {f'{with_avg_trials:.2f}':<25} │ {f'{without_avg_trials:.2f}':<23} │")
print(f"│ {'Average Score':<25} │ {f'{with_avg_score:.1f}/100':<25} │ {f'{without_avg_score:.1f}/100':<23} │")
print(f"{'└'+'─'*78+'┘'}\n")

# Calculate improvements
if without_avg_trials > 0:
    trials_improvement = ((without_avg_trials - with_avg_trials) / without_avg_trials) * 100
else:
    trials_improvement = 0

score_improvement = with_avg_score - without_avg_score

print(f"{'▶ REFLEXION IMPROVEMENTS:':<40}")
print(f"  • Trials reduction: {trials_improvement:.1f}%")
print(f"  • Score improvement: +{score_improvement:.1f} points")
print(f"  • Success rate change: {with_success}/{len(all_results_with)} vs {without_success}/{len(all_results_without)}")


In [ ]:
results_file = "reflexion_experiments_with_comparison.json"

output = {
    "experiment_date": datetime.now().isoformat(),
    "model": config_loader.model_name,
    "comparison_results": {
        "with_reflection": all_results_with,
        "without_reflection": all_results_without
    },
    "summary": {
        "with_reflection": {
            "success_rate": with_success / len(all_results_with) if all_results_with else 0,
            "avg_trials": with_avg_trials,
            "avg_score": with_avg_score
        },
        "without_reflection": {
            "success_rate": without_success / len(all_results_without) if all_results_without else 0,
            "avg_trials": without_avg_trials,
            "avg_score": without_avg_score
        },
        "improvements": {
            "trials_reduction_percent": trials_improvement,
            "score_improvement": score_improvement
        }
    },
    "key_finding": f"Self-Reflection improves performance by {trials_improvement:.1f}% fewer trials and {score_improvement:.1f} point score gain!"
}

with open(results_file, 'w') as f:
    json.dump(output, f, indent=2)

print(f"\n✅ Results saved to {results_file}")
print(f"✅ Episodic memory saved (shows learned reflections)")
print(f"\n{'='*80}")
print("🎉 REFLEXION FRAMEWORK - OBSERVABLE & COMPARABLE IMPLEMENTATION COMPLETE!")
print("="*80)

print(f"\n📊 KEY INSIGHT:")
print(f"The verbal self-reflection mechanism CLEARLY shows:")
print(f"  ✓ Full reflection text printed each failure")
print(f"  ✓ Episodic memory evolution displayed")
print(f"  ✓ WITH reflection: {with_avg_trials:.1f} avg trials, {with_avg_score:.1f}/100 avg score")
print(f"  ✓ WITHOUT reflection: {without_avg_trials:.1f} avg trials, {without_avg_score:.1f}/100 avg score")
print(f"  ✓ Improvement: {trials_improvement:.1f}% fewer trials needed!\n")
